# RSI + Bollinger Band Strategy
_Assess both the momentum and volatility of a price_

### Import Library + Data

In [43]:
# Import libraries
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set our time window (factor in the 50 day waiting period for SMA to kick in)
start_date = '2020-10-12'
end_date = '2021-12-30'
 
# Type in any ticker to analyze its behavior against our SMA strategy
ticker = 'TSLA'
df = yf.download(ticker, start_date, end_date)


# For our purposes, we only want the closing prices
df = df[['Close']].dropna()

# 20-day Simple Moving Average
df['SMA'] = df['Close'].rolling(window=20).mean()

# 20-day Standard Deviation
df['STDDEV'] = df['Close'].rolling(window=20).std()

# Upper and Lower Bollinger Bands
df['Upper_Band'] = df['SMA'] + (2 * df['STDDEV'])
df['Lower_Band'] = df['SMA'] - (2 * df['STDDEV'])

[*********************100%***********************]  1 of 1 completed


### Calculate RSI

In [44]:
# Price change from previous close
delta = df['Close'].diff()

# Positive gains (clip negatives to 0)
gain = delta.clip(lower=0)

# Negative losses (clip positives to 0, take negative)
loss = -delta.clip(upper=0)

# 14-day averages of gain/loss
avg_gain = gain.rolling(window=14, min_periods=14).mean()
avg_loss = loss.rolling(window=14, min_periods=14).mean()

# Relative Strength
rs = avg_gain / avg_loss

# RSI calculation
df['RSI'] = 100 - (100 / (1 + rs))

df = df.iloc[20:]

df.head(20)

Price,Close,SMA,STDDEV,Upper_Band,Lower_Band,RSI
Ticker,TSLA,,,,,
Date,,,,,,
2020-11-09,140.419998,141.714499,5.628108,152.970716,130.458282,49.743857
2020-11-10,136.786667,141.109666,5.464774,152.039214,130.180117,45.704485
2020-11-11,139.043335,140.373499,4.591999,149.557498,131.189501,47.045588
2020-11-12,137.253326,139.754832,4.085289,147.925411,131.584254,46.978253
2020-11-13,136.166672,139.235332,3.827287,146.889907,131.580758,46.064940
2020-11-16,136.029999,138.856332,3.745721,146.347774,131.364891,44.306408
2020-11-17,147.203339,139.184166,4.173196,147.530559,130.837773,61.083777
2020-11-18,162.213333,140.250832,6.631690,153.514212,126.987453,68.879810


### Buy/Sell Conditions

In [45]:
# First create neutral Signals
df['Signal'] = 0

# Buy when Buy Condition becomes newly true
buy_signal = (df['RSI'] < 30) & (df['Close'] <= df['Lower_Band'])
sell_signal = (df['RSI'] > 70) & (df['Close'] >= df['Upper_Band'])

# Entry only on transition
df.loc[(buy_signal) & (~buy_signal.shift(1)), 'Signal'] = 1
df.loc[(sell_signal) & (~sell_signal.shift(1)), 'Signal'] = -1

# Forward fill Position
df['Position'] = df['Signal'].replace(to_replace=0, method='ffill')


ValueError: Operands are not aligned. Do `left, right = left.align(right, axis=1, copy=False)` before operating.